# Fermi-Hubbard Model

The Fermi-Hubbard model is one of the simplest and most important models
used to study strongly correlated electrons in a lattice. Unlike the
independent-electron approximation, this model considers the competition
between the motion of electrons and their mutual interactions.

The main physics arises from two competing processes. The first is electron
hopping between neighboring lattice sites, which is associated with kinetic
energy and is controlled by the hopping parameter $t$. The second is the
Coulomb interaction between electrons, controlled by the interaction
parameter $U$.

The Hamiltonian is written as:

$$
H = H_t + H_U
$$

where the hopping term is:

$$
H_t = -t \sum_{\sigma=\uparrow,\downarrow}
\left(
f_{1\sigma}^{\dagger}f_{2\sigma}
+
f_{2\sigma}^{\dagger}f_{1\sigma}
\right)
$$

This term describes an electron with spin $\sigma$ moving between the two
lattice sites. Here, $f^\dagger$ and $f$ are the fermionic creation and
annihilation operators, respectively.

The interaction term is:

$$
H_U = U \sum_{i=1}^{2} n_{i\uparrow}n_{i\downarrow}
$$

This term represents the energy cost when electrons with opposite spins
occupy the same lattice site. When $U$ is large compared with $t$, electron
repulsion becomes more important and strong correlation effects can emerge.

Therefore, the competition between electron hopping and Coulomb repulsion
provides a controlled framework for studying interaction-driven quantum
phenomena. The Fermi-Hubbard model is particularly relevant to Mott
insulators, magnetic correlations, quantum phase transitions, and models
related to high-temperature superconductivity.

To simulate this Hamiltonian on a quantum computer, the fermionic operators
must first be mapped onto qubit operators. A Jordan-Wigner transformation
provides one such mapping, converting the fermionic terms into combinations
of Pauli operators that can be implemented using quantum gates.rs that can be implemented on qubits.

### PAULI-BASIS MEASUREMENTS for Fermi-Hubbard Model

In [ ]:
import numpy as np
from numpy import pi
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel="ibm_quantum_platform")
backend = service.backend("ibm_fez")

shots = #type in the number of shots depending up on your problem

total_time = # evolution time
iterations = # evolution iteration
dt = total_time / iterations

#Fermi Hubbard Parameters
t = # hopping strength
u = # onsite interaction

print("Backend:", backend.name)
print("dt per Trotter step:", dt)

def fermihubbard_circuit(n, t, u, dt):
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.h(1)

    for _ in range(n):

        qc.h(0)
        qc.h(1)
        qc.cx(0, 1)
        qc.rz(-2 * t * dt, 1)
        qc.cx(0, 1)
        qc.h(0)
        qc.h(1)

        qc.sdg(0)
        qc.sdg(1)
        qc.h(0)
        qc.h(1)
        qc.cx(0, 1)
        qc.rz(-2 * t * dt, 1)
        qc.cx(0, 1)
        qc.h(0)
        qc.h(1)
        qc.s(0)
        qc.s(1)

        qc.cx(0, 1)
        qc.rz(-2 * u * dt, 1)
        qc.cx(0, 1)

    return qc

# BASIS ROTATIONS + MEASUREMENT
def add_basis_and_measure(qc, basis):
    qc2 = qc.copy()
    for q, b in enumerate(basis):
        if b == "X":
            qc2.h(q)
        elif b == "Y":
            qc2.sdg(q)
            qc2.h(q)
    qc2.measure_all()
    return qc2

bases = ["ZZ","ZX","ZY","ZI",
         "XZ","XX","XY","XI",
         "YZ","YX","YY","YI",
         "IZ","IX","IY"] # Measurement Bases

all_circuits = []

for n in range(1, iterations + 1):
    for basis in bases:
        base = fermihubbard_circuit(n, t, u, dt)
        full = add_basis_and_measure(base, basis)
        all_circuits.append(full)

print("Total circuits submitted:", len(all_circuits)) #Total 75 Circuits

# TRANSPILATION + EXECUTION
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pm.run(all_circuits)

sampler = Sampler(mode=backend)
job = sampler.run(isa_circuits, shots=shots)

print("\nSAVE THIS JOB ID:")
print(job.job_id())

## Retrieving the Job Results and Performing State Analysis

After submitting the quantum circuits to the IBM Quantum backend, the Job ID
printed by the execution code should be saved. The Job ID uniquely identifies
the submitted experiment and allows the experimental results to be retrieved
later without executing the quantum circuits again.

The saved Job ID is entered into the following analysis code:

```python
job = service.job("YOUR_JOB_ID")